In [6]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

clinical = pd.read_csv('dataset/clinical.csv')
clinical = clinical.drop(['id', 'creation_datetime', 'original_patientID', 'OS_status', 'OS_month', 'immunotherapy_treatment', 'CNA_data', 'SNV_data', 'GEX_data'], axis=1)
clinical = clinical.drop(['patientID', 'AJCC_stage', 'pre_MAPKi_treatment'], axis=1)
clinical.head()

,sex,age,M_stage,LDH,PFS_status,PFS_month,drug,BOR,BRAF_mut,brain_metastasis,source
0,male,56,NaN,normal,1.0,30.5,dabrafenib + trametinib,PD,V600E,no,doi:10.3390/cancers12082224
1,male,86,NaN,normal,0.0,24.1,dabrafenib,SD,V600E,no,doi:10.3390/cancers12082224
2,female,47,NaN,normal,0.0,14.1,dabrafenib + trametinib,CR,V600E,no,doi:10.3390/cancers12082224
3,female,50,NaN,NaN,1.0,1.6,vemurafenib,PD,V600E,no,doi:10.3390/cancers12082224
4,female,47,NaN,elevated,1.0,11.9,dabrafenib + trametinib,PD,V600K,no,doi:10.3390/cancers12082224


In [7]:
clinical = clinical.dropna(subset=['BOR'])

In [8]:
clinical.info()

<class 'pandas.core.frame.DataFrame'>
Index: 410 entries, 0 to 416
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sex               410 non-null    object 
 1   age               410 non-null    int64  
 2   M_stage           276 non-null    object 
 3   LDH               269 non-null    object 
 4   PFS_status        410 non-null    float64
 5   PFS_month         410 non-null    float64
 6   drug              409 non-null    object 
 7   BOR               410 non-null    object 
 8   BRAF_mut          410 non-null    object 
 9   brain_metastasis  235 non-null    object 
 10  source            410 non-null    object 
dtypes: float64(2), int64(1), object(8)
memory usage: 38.4+ KB


In [9]:
clinical['BOR_binary'] = clinical['BOR'].map({'CR': 1, 'PR': 1, 'SD': 0, 'PD': 0})
# y = clinical['BOR_binary']

def get_pfs_label(row):
    months = row['PFS_month']
    event = row['PFS_status']
    
    if months >= 18:
        return 2  # Long responder
    elif months < 6 and event == 1:
        return 0   # Non responder
    elif 6 <= months < 18 and event == 1:
        return 1 # Intermediate
    else:
        return np.nan   # Undetermind
    
def get_bin_pfs_label(row):
    months = row['PFS_month']
    event = row['PFS_status']
    
    if months >= 6:
        return 1  # Responder
    elif months < 6 and event == 1:
        return 0   # Non responder
    # elif 6 <= months < 18 and event == 1:
    #     return 1 # Intermediate
    else:
        return np.nan   # Undetermind
    
clinical['pfs_label'] = clinical.apply(get_pfs_label, axis=1)
clinical = clinical.dropna(subset=['pfs_label']).copy()
y = clinical['pfs_label']

X = clinical.drop(['pfs_label', 'PFS_status', 'PFS_month', 'BOR', 'BOR_binary', 'source'], axis=1)
X

,sex,age,M_stage,LDH,drug,BRAF_mut,brain_metastasis
0,male,56,NaN,normal,dabrafenib + trametinib,V600E,no
1,male,86,NaN,normal,dabrafenib,V600E,no
3,female,50,NaN,NaN,vemurafenib,V600E,no
4,female,47,NaN,elevated,dabrafenib + trametinib,V600K,no
5,female,53,NaN,elevated,vemurafenib,V600E,yes
...,...,...,...,...,...,...,...
412,male,47,M1C,NaN,vemurafenib,V600E,no
413,male,39,M1A,NaN,vemurafenib,V600E,no
414,male,84,M1C,NaN,dabrafenib,V600E,no
415,female,41,M1C,NaN,vemurafenib,V600E,no


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest, f_classif, RFECV, VarianceThreshold
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_validate, LeaveOneGroupOut, cross_val_predict
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from tqdm import tqdm
from xgboost import XGBClassifier

m_stage_prep = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[['M1A', 'M1B', 'M1C']],
        handle_unknown='use_encoded_value',
        unknown_value=np.nan
    )),
])

other_feat_prep = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('m_stage', m_stage_prep, ['M_stage']),
    ('other_cat_feat', other_feat_prep, ['sex', 'LDH', 'drug', 'BRAF_mut', 'brain_metastasis'])
], remainder='passthrough')
# np.set_printoptions(threshold=np.inf)

groups = clinical['source']
logo = LeaveOneGroupOut()

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'SVM': SVC(kernel='linear'),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

for name, model in tqdm(models.items(), desc='Models'):
    
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    cv_results = cross_validate(
        estimator=pipeline,
        X=X,
        y=y,
        cv=logo,
        groups=groups,
        scoring='f1_macro',
        return_train_score=True, 
        n_jobs=-1
    )
    
    cv_results = cross_validate(pipeline, X, y, cv=logo, groups=groups, scoring='f1_macro')
    mean = cv_results['test_score'].mean()
    std = cv_results['test_score'].std()
    print(f"\n{name}: {mean:.3f} ± {std:.3f}")
    
    y_pred = cross_val_predict(pipeline, X, y, cv=logo, groups=groups)
    print(classification_report(y, y_pred))
    

# cohort_sizes = [groups.iloc[test_idx].shape[0] for _, test_idx in logo.split(X, y, groups)]

# weighted_score = np.average(cv_results['test_score'], weights=cohort_sizes)
# print("Per-cohort scores:", cv_results['test_score'])
# print("Weighted mean F1:", weighted_score)

# print(cv_results['test_score'])
# print(cv_results['test_score'].mean())
# print(cv_results['test_score'].std())

Models:   0%|          | 0/6 [00:00<?, ?it/s]


LogisticRegression: 0.403 ± 0.077


Models:  17%|█▋        | 1/6 [00:01<00:08,  1.68s/it]

              precision    recall  f1-score   support

         0.0       0.66      0.78      0.71       207
         1.0       0.45      0.36      0.40       118
         2.0       0.24      0.12      0.16        32

    accuracy                           0.58       357
   macro avg       0.45      0.42      0.43       357
weighted avg       0.55      0.58      0.56       357


SVM: 0.386 ± 0.131


/Users/buudinhha/PycharmProjects/internship-m2-melanoma/main/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/buudinhha/PycharmProjects/internship-m2-melanoma/main/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/buudinhha/PycharmProjects/internship-m2-melanoma/main/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `

...

              precision    recall  f1-score   support

         0.0       0.62      0.62      0.62       207
         1.0       0.39      0.36      0.37       118
         2.0       0.14      0.19      0.16        32

    accuracy                           0.50       357
   macro avg       0.39      0.39      0.39       357
weighted avg       0.50      0.50      0.50       357


GradientBoosting: 0.352 ± 0.062


Models:  83%|████████▎ | 5/6 [00:09<00:02,  2.43s/it]

              precision    recall  f1-score   support

         0.0       0.61      0.60      0.61       207
         1.0       0.46      0.38      0.42       118
         2.0       0.12      0.22      0.16        32

    accuracy                           0.49       357
   macro avg       0.40      0.40      0.39       357
weighted avg       0.52      0.49      0.50       357


XGBoost: 0.343 ± 0.062


Models: 100%|██████████| 6/6 [00:16<00:00,  2.83s/it]

              precision    recall  f1-score   support

         0.0       0.62      0.60      0.61       207
         1.0       0.40      0.37      0.38       118
         2.0       0.13      0.19      0.16        32

    accuracy                           0.49       357
   macro avg       0.38      0.39      0.38       357
weighted avg       0.50      0.49      0.50       357



In [11]:
# from sklearn.model_selection import cross_val_predict
# from sklearn.metrics import classification_report

# y_pred = cross_val_predict(pipeline, X, y, cv=logo, groups=groups)
# print(classification_report(y, y_pred))


In [12]:
# from sklearn.dummy import DummyClassifier
# dummy = DummyClassifier(strategy='most_frequent')
# dummy_results = cross_val_predict(dummy, X, y, cv=logo, groups=groups)
# print(classification_report(y, dummy_results))


In [13]:
# full_pipeline.fit(X, y)
# feature_names = full_pipeline.named_steps['preprocessor'].get_feature_names_out()
# importances = full_pipeline.named_steps['model'].feature_importances_

# for name, importance in sorted(zip(feature_names, importances), key=lambda x: x[1], reverse=True):
#     print(f"{name}: {importance:.4f}")
